# Smart Disaster Command Center PoC

This notebook builds the district backbone, converts the flood and weather datasets into district-level telemetry, computes deterministic flood/heat flags, and generates Qwen-based explanations for the dashboard.

The pipeline follows the assignment requirements:
- exactly two disasters: Floods and Heatwaves
- localized mock telemetry across districts
- grounded AI assistant that answers only from the generated dataset
- clean single-page dashboard artifacts for Streamlit

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
#for dirname, _, filenames in os.walk('/kaggle/input'):
 #   for filename in filenames:
 #       print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# **Preprocessing Dataset and handling naming mismatches**

In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np
import re
from pathlib import Path

In [3]:
CENSUS_CSV = "/kaggle/input/datasets/shiivvvaam/indian-districts-population-data/census2011.csv"

DISTRICT_SHP = "/kaggle/input/datasets/anshpatidar/india-districts-geojson/Districts/Census_2011/2011_Dist.shp"

ART = Path("/kaggle/working/artifacts")
ART.mkdir(exist_ok=True)

In [4]:
census = pd.read_csv(CENSUS_CSV)

geo = gpd.read_file(DISTRICT_SHP)

print(census.shape)
print(geo.shape)

display(census.head())
display(geo.head())

(610, 7)
(641, 6)


,Ranking,District,State,Population,Growth,Sex-Ratio,Literacy
0,1,Thane,Maharashtra,"11,060,148",36.01 %,886,84.53
1,2,North Twenty Four Parganas,West Bengal,"10,009,781",12.04 %,955,84.06
2,3,Bangalore,Karnataka,"9,621,551",47.18 %,916,87.67
3,4,Pune,Maharashtra,"9,429,408",30.37 %,915,86.15
4,5,Mumbai Suburban,Maharashtra,"9,356,962",8.29 %,860,89.91


,DISTRICT,ST_NM,ST_CEN_CD,DT_CEN_CD,censuscode,geometry
0,Adilabad,Andhra Pradesh,28,1,532,"POLYGON ((78.84972 19.7601, 78.85102 19.75945,..."
1,Agra,Uttar Pradesh,9,15,146,"POLYGON ((78.19803 27.4028, 78.19804 27.40278,..."
2,Ahmadabad,Gujarat,24,7,474,"MULTIPOLYGON (((72.03456 23.50527, 72.03337 23..."
3,Ahmadnagar,Maharashtra,27,26,522,"POLYGON ((74.67333 19.9467, 74.67393 19.93509,..."
4,Aizawl,Mizoram,15,3,283,"POLYGON ((92.98749 24.40453, 92.99107 24.40236..."


In [5]:
census.columns = (
    census.columns
    .str.strip()
    .str.lower()
    .str.replace("-", "_")
)

census.columns

Index(['ranking', 'district', 'state', 'population', 'growth', 'sex_ratio',
       'literacy'],
      dtype='object')

In [6]:
def clean_text(x):

    if pd.isna(x):
        return ""

    x = str(x).upper().strip()

    x = x.replace("&","AND")

    x = re.sub(r"[^A-Z0-9 ]"," ",x)

    x = re.sub(r"\s+"," ",x)

    return x

In [7]:
census["district_key"] = census["district"].map(clean_text)
census["state_key"] = census["state"].map(clean_text)

geo["district_key"] = geo["DISTRICT"].map(clean_text)
geo["state_key"] = geo["ST_NM"].map(clean_text)

In [8]:
district_fix = {

    "NORTH TWENTY FOUR PARGANAS":"NORTH 24 PARGANAS",
    "SOUTH TWENTY FOUR PARGANAS":"SOUTH 24 PARGANAS",

    "PURBI CHAMPARAN":"PURBA CHAMPARAN",

    "KANSHIRAM NAGAR":"KASGANJ",

    "RAMABAI NAGAR":"KANPUR DEHAT",

    "PASCHIM MEDINIPUR":"WEST MIDNAPORE",

    "PANCHMAHAL":"PANCH MAHALS",

    "BAUDH":"BOUDH",

    "CHHATTARPUR":"CHHATARPUR",

    "BANDIPORA":"BANDIPORE",

    "KABIRDHAM":"KABEERDHAM",

    "KOREA":"KORIYA",

    "MOHALI":"SAHIBZADA AJIT SINGH NAGAR",

    "MUMBAI CITY":"GREATER MUMBAI",

    "LAHUL AND SPITI":"LAHUL SPITI",

    "NICOBARS":"NICOBAR",

    "MORIGAON":"MARIGAON"
}

census["district_key"] = census["district_key"].replace(district_fix)

In [9]:
state_fix = {

    "ORISSA":"ODISHA",

    "JAMMU AND KASHMIR":"JAMMU KASHMIR",

    "ANDAMAN AND NICOBAR ISLANDS":"ANDAMAN NICOBAR"
}

census["state_key"] = census["state_key"].replace(state_fix)

geo["state_key"] = geo["state_key"].replace(state_fix)

In [10]:
district_master = census.merge(
    geo[
        [
            "district_key",
            "state_key",
            "DISTRICT",
            "ST_NM",
            "geometry"
        ]
    ],
    on=["district_key","state_key"],
    how="left"
)

In [11]:
district_master = district_master.drop_duplicates(
    subset=["district","state"]
)

print("Rows:", len(district_master))

Rows: 610


In [12]:
matched = district_master[
    district_master["geometry"].notna()
].copy()

matched_geo = gpd.GeoDataFrame(
    matched,
    geometry="geometry",
    crs="EPSG:4326"
)

proj = matched_geo.to_crs(3857)

cent = proj.centroid

cent = gpd.GeoSeries(
    cent,
    crs=3857
).to_crs(4326)

matched_geo["latitude"] = cent.y
matched_geo["longitude"] = cent.x

In [13]:
csv_out = matched_geo.drop(
    columns="geometry"
)

csv_out.to_csv(
    ART / "district_master.csv",
    index=False
)

print("Saved CSV")

Saved CSV


In [14]:
geojson_cols = []

for c in matched_geo.columns:

    if c not in geojson_cols:
        geojson_cols.append(c)

matched_geo = matched_geo[geojson_cols]

matched_geo.to_file(
    ART / "district_master.geojson",
    driver="GeoJSON"
)

print("Saved GeoJSON")

Saved GeoJSON


In [15]:
unmatched = district_master[
    district_master["geometry"].isna()
]

print("UNMATCHED:", len(unmatched))

display(
    unmatched[
        ["district","state"]
    ].sort_values("state")
)

unmatched.to_csv(
    ART / "unmatched_districts.csv",
    index=False
)

UNMATCHED: 54


,district,state
553,South Andaman,Andaman and Nicobar Islands
605,Nicobars,Andaman and Nicobar Islands
583,North and Middle Andaman,Andaman and Nicobar Islands
609,Dibang Valley,Arunachal Pradesh
608,Anjaw,Arunachal Pradesh
563,Papumpare,Arunachal Pradesh
567,Changlang,Arunachal Pradesh
569,Lohit,Arunachal Pradesh
581,West Siang,Arunachal Pradesh
582,Tirap,Arunachal Pradesh


In [16]:
print(
    "Duplicate districts:",
    matched_geo.duplicated(
        subset=["district","state"]
    ).sum()
)

print(
    "Missing geometry:",
    matched_geo.geometry.isna().sum()
)

print(
    "Final rows:",
    len(matched_geo)
)

display(
    matched_geo[
        [
            "district",
            "state",
            "latitude",
            "longitude"
        ]
    ].head()
)

Duplicate districts: 0
Missing geometry: 0
Final rows: 556


,district,state,latitude,longitude
0,Thane,Maharashtra,19.599899,73.150745
1,North Twenty Four Parganas,West Bengal,22.467110,88.780096
2,Bangalore,Karnataka,12.950945,77.595924
3,Pune,Maharashtra,18.571959,74.068686
4,Mumbai Suburban,Maharashtra,19.129627,72.885060


# **Combining Flood and Climate Change dataset with district IDs**

In [17]:
import os
import re
import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

In [18]:
# -----------------------------
# Paths
# -----------------------------
ART = Path("/kaggle/working/artifacts")
ART.mkdir(parents=True, exist_ok=True)

FLOOD_PATH = "/kaggle/input/datasets/s3programmer/flood-risk-in-india/flood_risk_dataset_india.csv"
WEATHER_PATH = "/kaggle/input/datasets/rosemeenshaikh/india-weather-dataset-2020-2025/popular_cities_weather.csv"
DISTRICT_GEOJSON = "/kaggle/working/artifacts/district_master.geojson"   # final district backbone you already created

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [19]:
flood = pd.read_csv(FLOOD_PATH)
weather = pd.read_csv(WEATHER_PATH)
districts = gpd.read_file(DISTRICT_GEOJSON)

print("Flood:", flood.shape)
print("Weather:", weather.shape)
print("Districts:", districts.shape)

Flood: (10000, 14)
Weather: (7056, 9)
Districts: (556, 14)


In [20]:
# normalize only what is needed, no extra columns
flood = flood.rename(columns={
    "Latitude": "latitude",
    "Longitude": "longitude",
    "Rainfall (mm)": "rainfall_mm",
    "Temperature (°C)": "temperature_c",
    "Humidity (%)": "humidity_",
    "River Discharge (m³/s)": "river_discharge_m3_s",
    "Water Level (m)": "water_level_m",
    "Elevation (m)": "elevation_m",
    "Land Cover": "land_cover",
    "Soil Type": "soil_type",
    "Population Density": "population_density",
    "Infrastructure": "infrastructure",
    "Historical Floods": "historical_floods",
    "Flood Occurred": "flood_occurred"
})

weather = weather.rename(columns={
    "date": "date",
    "tavg": "tavg",
    "tmin": "tmin",
    "tmax": "tmax",
    "prcp": "prcp",
    "wspd": "wspd",
    "pres": "pres",
    "tsun": "tsun",
    "city": "city"
})

# keep only important columns
flood = flood[[
    "latitude",
    "longitude",
    "rainfall_mm",
    "river_discharge_m3_s",
    "water_level_m",
    "elevation_m",
    "land_cover",
    "soil_type",
    "historical_floods",
    "flood_occurred"
]].copy()

weather = weather[[
    "date",
    "tavg",
    "tmax",
    "prcp",
    "city"
]].copy()

weather["date"] = pd.to_datetime(weather["date"])

In [21]:
# district geojson must already be the cleaned final artifact
districts.columns = [c.lower() for c in districts.columns]

# keep only important spatial columns
districts = districts[["district", "st_nm", "geometry"]].copy()

# remove any accidental duplicates
districts = districts.drop_duplicates(subset=["district", "st_nm"]).reset_index(drop=True)

# create district ids
districts["district_id"] = [f"D{i:04d}" for i in range(1, len(districts) + 1)]

# centroids for dashboard map/metrics
districts = districts.to_crs(epsg=4326)
districts["latitude"] = districts.geometry.centroid.y
districts["longitude"] = districts.geometry.centroid.x

print("District rows:", len(districts))
print("Duplicate district/state pairs:", districts.duplicated(subset=["district", "st_nm"]).sum())
print("Missing geometry:", districts.geometry.isna().sum())

display(districts.head())

District rows: 556
Duplicate district/state pairs: 0
Missing geometry: 0


/tmp/ipykernel_23/496439834.py:15: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  districts["latitude"] = districts.geometry.centroid.y
/tmp/ipykernel_23/496439834.py:16: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  districts["longitude"] = districts.geometry.centroid.x


,district,district,st_nm,geometry,district_id,latitude,longitude
0,Thane,Thane,Maharashtra,"MULTIPOLYGON (((72.87769 20.2264, 72.88886 20....",D0001,19.599155,73.150900
1,North Twenty Four Parganas,North 24 Parganas,West Bengal,"MULTIPOLYGON (((88.82434 23.24874, 88.8283 23....",D0002,22.464952,88.780323
2,Bangalore,Bangalore,Karnataka,"POLYGON ((77.83549 12.86809, 77.83213 12.86372...",D0003,12.950835,77.595936
3,Pune,Pune,Maharashtra,"POLYGON ((74.83388 18.3392, 74.83266 18.32908,...",D0004,18.570885,74.069082
4,Mumbai Suburban,Mumbai Suburban,Maharashtra,"POLYGON ((72.95859 18.98734, 72.94677 18.98574...",D0005,19.129580,72.885070


In [22]:
# flood points to geometry
flood_gdf = gpd.GeoDataFrame(
    flood,
    geometry=gpd.points_from_xy(flood["longitude"], flood["latitude"]),
    crs="EPSG:4326"
)

# spatial join against the district polygons
flood_join = gpd.sjoin(
    flood_gdf,
    districts[["district_id", "district", "st_nm", "geometry"]],
    how="left",
    predicate="within"
)

print("Flood join shape:", flood_join.shape)
print("Matched flood rows:", flood_join["district_id"].notna().sum())
print("Unmatched flood rows:", flood_join["district_id"].isna().sum())

display(flood_join[["district", "st_nm", "rainfall_mm", "water_level_m"]].head())

Flood join shape: (10000, 16)
Matched flood rows: 2833
Unmatched flood rows: 7167


,district,district,st_nm,rainfall_mm,water_level_m
0,Karimnagar,Karimnagar,Andhra Pradesh,218.999493,7.415552
1,NaN,NaN,NaN,55.353599,8.811019
2,Ganganagar,Ganganagar,Rajasthan,103.991908,4.631799
3,NaN,NaN,NaN,198.984191,2.891787
4,NaN,NaN,NaN,144.626803,3.188466


In [23]:
flood_join.loc[:, flood_join.columns=="district"].head()

,district,district
0,Karimnagar,Karimnagar
1,NaN,NaN
2,Ganganagar,Ganganagar
3,NaN,NaN
4,NaN,NaN


In [24]:
flood_join = gpd.sjoin(
    flood_gdf,
    districts[["district_id","district","st_nm","geometry"]],
    how="left",
    predicate="within"
)

In [25]:
# show duplicates
print(
    flood_join.columns[
        flood_join.columns.duplicated()
    ]
)

Index(['district'], dtype='object')


In [26]:
# keep only first occurrence of duplicated columns
flood_join = flood_join.loc[
    :,
    ~flood_join.columns.duplicated()
]

In [27]:
district_flood = (
    flood_join
    .dropna(subset=["district_id"])
    .groupby(["district_id", "district", "st_nm"], as_index=False)
    .agg(
        avg_rainfall=("rainfall_mm", "mean"),
        max_rainfall=("rainfall_mm", "max"),
        avg_discharge=("river_discharge_m3_s", "mean"),
        max_discharge=("river_discharge_m3_s", "max"),
        avg_water_level=("water_level_m", "mean"),
        max_water_level=("water_level_m", "max"),
        avg_elevation=("elevation_m", "mean"),
        historical_flood_ratio=("historical_floods", "mean"),
        flood_rate=("flood_occurred", "mean"),
        sample_count=("flood_occurred", "count")
    )
)

# deterministic flood risk
district_flood["flood_risk_score"] = (
    0.35 * district_flood["flood_rate"] +
    0.20 * (district_flood["avg_rainfall"] / (district_flood["avg_rainfall"].max() + 1e-6)) +
    0.20 * (district_flood["avg_discharge"] / (district_flood["avg_discharge"].max() + 1e-6)) +
    0.15 * (district_flood["avg_water_level"] / (district_flood["avg_water_level"].max() + 1e-6)) +
    0.10 * district_flood["historical_flood_ratio"]
) * 100

district_flood["flood_risk_score"] = district_flood["flood_risk_score"].clip(0, 100)

display(district_flood.head())

,district_id,district,st_nm,avg_rainfall,max_rainfall,avg_discharge,max_discharge,avg_water_level,max_water_level,avg_elevation,historical_flood_ratio,flood_rate,sample_count,flood_risk_score
0,D0001,Thane,Maharashtra,110.568991,248.571944,2234.896506,4906.385206,4.993047,9.980659,5006.417812,0.411765,0.647059,17,50.980522
1,D0002,North Twenty Four Parganas,West Bengal,134.075355,227.935836,2054.764850,4944.233976,4.255246,8.404700,3904.018959,0.571429,0.714286,7,54.675497
2,D0003,Bangalore,Karnataka,74.645162,109.321412,2303.918771,4065.758959,4.894393,8.162394,2475.834045,0.500000,0.000000,2,26.890529
3,D0004,Pune,Maharashtra,159.060003,264.747938,2221.524900,4822.149942,3.498621,9.539799,4091.783354,0.473684,0.421053,19,44.649054
4,D0006,South Twenty Four Parganas,West Bengal,66.781810,168.774316,4007.096509,4967.938254,6.639614,9.313039,5621.822645,0.750000,0.750000,4,64.643128


In [28]:
district_cols = [
    i for i,c in enumerate(districts.columns)
    if c=="district"
]

print(district_cols)

for idx in district_cols:
    print("\nCOLUMN POSITION:", idx)
    display(
        districts.iloc[:, [idx]].head()
    )

[0, 1]

COLUMN POSITION: 0


,district
0,Thane
1,North Twenty Four Parganas
2,Bangalore
3,Pune
4,Mumbai Suburban



COLUMN POSITION: 1


,district
0,Thane
1,North 24 Parganas
2,Bangalore
3,Pune
4,Mumbai Suburban


In [29]:
# keep first occurrence of duplicate names

districts = districts.loc[
    :,
    ~districts.columns.duplicated()
].copy()

print(districts.columns.tolist())

['district', 'st_nm', 'geometry', 'district_id', 'latitude', 'longitude']


In [30]:
print("Duplicate columns:",
      districts.columns.duplicated().sum())

print(
    districts.columns[
        districts.columns.duplicated()
    ]
)

Duplicate columns: 0
Index([], dtype='object')


In [31]:
def norm_text(x):
    if pd.isna(x):
        return ""
    x = str(x).strip().upper()
    x = re.sub(r"[^A-Z0-9 ]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

# district lookup
district_lookup = {norm_text(d): d for d in districts["district"].unique()}

# small alias table for common mismatches
CITY_ALIAS_TO_DISTRICT = {
    "MUMBAI": "MUMBAI SUBURBAN",
    "NEW DELHI": "NEW DELHI",
    "DELHI": "CENTRAL",
    "BENGALURU": "BANGALORE",
    "BANGALORE": "BANGALORE",
    "KOLKATA": "KOLKATA",
    "CHENNAI": "CHENNAI",
    "HYDERABAD": "HYDERABAD",
    "PUNE": "PUNE",
    "NAGPUR": "NAGPUR",
    "AHMEDABAD": "AHMADABAD",
    "JAIPUR": "JAIPUR",
    "LUCKNOW": "LUCKNOW",
    "BHOPAL": "BHOPAL",
    "INDORE": "INDORE",
    "SURAT": "SURAT",
    "VADODARA": "VADODARA",
    "COIMBATORE": "COIMBATORE",
    "VISAKHAPATNAM": "VISAKHAPATNAM",
    "THIRUVANANTHAPURAM": "THIRUVANANTHAPURAM",
    "KOCHI": "ERNAKULAM"
}

def city_to_district(city):
    key = norm_text(city)
    if key in CITY_ALIAS_TO_DISTRICT:
        return CITY_ALIAS_TO_DISTRICT[key]
    if key in district_lookup:
        return district_lookup[key]
    return None

weather["district"] = weather["city"].map(city_to_district)

# keep only matched rows for district-level heat features
weather_matched = weather.dropna(subset=["district"]).copy()

print("Weather matched rows:", len(weather_matched))
print("Unmatched city rows:", weather["district"].isna().sum())

display(weather_matched.head())

Weather matched rows: 5112
Unmatched city rows: 1944


,date,tavg,tmax,prcp,city,district
0,2020-01-01,24.9,29.9,0.0,Mumbai,MUMBAI SUBURBAN
1,2020-02-01,27.3,32.6,0.0,Mumbai,MUMBAI SUBURBAN
2,2020-03-01,27.7,31.9,0.0,Mumbai,MUMBAI SUBURBAN
3,2020-04-01,30.2,33.9,0.0,Mumbai,MUMBAI SUBURBAN
4,2020-05-01,31.1,34.0,0.0,Mumbai,MUMBAI SUBURBAN


In [32]:
# heat proxy / WBGT proxy
# simple deterministic proxy that is stable for dashboard use
weather_matched["wbgt_est"] = (
    0.65 * weather_matched["tavg"] +
    0.35 * weather_matched["tmax"] -
    0.05 * weather_matched["prcp"].clip(0, 20)
).clip(lower=0)

weather_matched["heat_tier"] = pd.cut(
    weather_matched["wbgt_est"],
    bins=[-1, 28, 30, 32, 100],
    labels=["Normal", "Yellow", "Orange", "Red"]
)

weather_matched["vulnerable_clusters"] = np.where(
    weather_matched["heat_tier"] == "Red", 5,
    np.where(weather_matched["heat_tier"] == "Orange", 3,
    np.where(weather_matched["heat_tier"] == "Yellow", 1, 0))
)

In [33]:
district_heat = (
    weather_matched
    .groupby(["district"], as_index=False)
    .agg(
        avg_temp=("tavg", "mean"),
        max_temp=("tmax", "max"),
        avg_prcp=("prcp", "mean"),
        avg_wbgt=("wbgt_est", "mean"),
        vulnerable_clusters=("vulnerable_clusters", "sum")
    )
)

district_heat["heat_risk_score"] = (
    0.45 * (district_heat["avg_temp"] / (district_heat["avg_temp"].max() + 1e-6)) +
    0.35 * (district_heat["max_temp"] / (district_heat["max_temp"].max() + 1e-6)) +
    0.20 * (district_heat["avg_wbgt"] / (district_heat["avg_wbgt"].max() + 1e-6))
) * 100

district_heat["heat_risk_score"] = district_heat["heat_risk_score"].clip(0, 100)

display(district_heat.head())

,district,avg_temp,max_temp,avg_prcp,avg_wbgt,vulnerable_clusters,heat_risk_score
0,AHMADABAD,28.250000,42.8,80.111111,29.955347,167,97.560713
1,Agra,25.176389,41.8,68.009259,26.922685,109,90.005992
2,Ajmer,25.100000,41.7,77.932143,26.968750,74,89.839536
3,Akola,27.500000,44.0,86.475000,29.160069,242,96.829010
4,Aligarh,24.548611,41.5,73.550000,26.010208,107,88.191126


In [34]:
district_final = districts.merge(
    district_flood,
    on=["district_id", "district", "st_nm"],
    how="left"
).merge(
    district_heat,
    on=["district"],
    how="left",
    suffixes=("", "_heat")
)

# fill missing values cleanly
for col in [
    "avg_rainfall", "max_rainfall", "avg_discharge", "max_discharge",
    "avg_water_level", "max_water_level", "avg_elevation",
    "historical_flood_ratio", "flood_rate", "sample_count",
    "flood_risk_score", "avg_temp", "max_temp", "avg_prcp",
    "avg_wbgt", "vulnerable_clusters", "heat_risk_score"
]:
    if col in district_final.columns:
        district_final[col] = district_final[col].fillna(0)

district_final["overall_risk_score"] = (
    0.55 * district_final["flood_risk_score"] +
    0.45 * district_final["heat_risk_score"]
).clip(0, 100)

# deterministic flags
district_final["water_level_above_danger_m"] = district_final["avg_water_level"]
district_final["active_flood_hotspots"] = np.where(
    (district_final["avg_rainfall"] > district_final["avg_rainfall"].median()) &
    (district_final["avg_water_level"] > district_final["avg_water_level"].median()),
    np.ceil((district_final["avg_rainfall"] / (district_final["avg_rainfall"].max() + 1e-6)) * 5).astype(int),
    0
)

district_final["response_teams_deployed"] = np.where(
    district_final["active_flood_hotspots"] > 0,
    np.maximum(1, np.ceil(district_final["active_flood_hotspots"] / 2)).astype(int),
    0
)

district_final["heat_alert_tier"] = pd.cut(
    district_final["avg_wbgt"],
    bins=[-1, 28, 30, 32, 100],
    labels=["Normal", "Yellow", "Orange", "Red"]
).astype(str)

district_final["flood_severity"] = pd.cut(
    district_final["water_level_above_danger_m"],
    bins=[-1, 0.5, 1.0, 1.5, 100],
    labels=["Low", "Moderate", "High", "Critical"]
).astype(str)

district_final["flood_risk_flag"] = district_final["water_level_above_danger_m"] > 0.5
district_final["heat_severity_flag"] = district_final["heat_alert_tier"].isin(["Yellow", "Orange", "Red"])
district_final["dual_hazard_flag"] = district_final["flood_risk_flag"] & district_final["heat_severity_flag"]

district_final["overall_severity"] = np.select(
    [
        (district_final["flood_severity"] == "Critical") | (district_final["heat_alert_tier"] == "Red"),
        (district_final["flood_severity"] == "High") | (district_final["heat_alert_tier"] == "Orange"),
        (district_final["flood_severity"] == "Moderate") | (district_final["heat_alert_tier"] == "Yellow")
    ],
    ["Extreme", "High", "Moderate"],
    default="Low"
)

print("Final rows:", len(district_final))
print("Duplicates:", district_final.duplicated(subset=["district", "st_nm"]).sum())
print("Missing geometry:", district_final.geometry.isna().sum())

display(district_final.head())

Final rows: 556
Duplicates: 0
Missing geometry: 0


,district,st_nm,geometry,district_id,latitude,longitude,avg_rainfall,max_rainfall,avg_discharge,max_discharge,...,overall_risk_score,water_level_above_danger_m,active_flood_hotspots,response_teams_deployed,heat_alert_tier,flood_severity,flood_risk_flag,heat_severity_flag,dual_hazard_flag,overall_severity
0,Thane,Maharashtra,"MULTIPOLYGON (((72.87769 20.2264, 72.88886 20....",D0001,19.599155,73.150900,110.568991,248.571944,2234.896506,4906.385206,...,68.978003,4.993047,0,0,Yellow,Critical,True,True,True,Extreme
1,North Twenty Four Parganas,West Bengal,"MULTIPOLYGON (((88.82434 23.24874, 88.8283 23....",D0002,22.464952,88.780323,134.075355,227.935836,2054.764850,4944.233976,...,30.071523,4.255246,0,0,Normal,Critical,True,False,False,Extreme
2,Bangalore,Karnataka,"POLYGON ((77.83549 12.86809, 77.83213 12.86372...",D0003,12.950835,77.595936,74.645162,109.321412,2303.918771,4065.758959,...,14.789791,4.894393,0,0,Normal,Critical,True,False,False,Extreme
3,Pune,Maharashtra,"POLYGON ((74.83388 18.3392, 74.83266 18.32908,...",D0004,18.570885,74.069082,159.060003,264.747938,2221.524900,4822.149942,...,24.556980,3.498621,0,0,Normal,Critical,True,False,False,Extreme
4,Mumbai Suburban,Maharashtra,"POLYGON ((72.95859 18.98734, 72.94677 18.98574...",D0005,19.129580,72.885070,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0,0,Normal,Low,False,False,False,Low


In [35]:
# CSV artifact
district_final.drop(columns=["geometry"]).to_csv(ART / "district_master_enriched.csv", index=False)

# GeoJSON artifact
district_final.to_file(ART / "district_master_enriched.geojson", driver="GeoJSON")

print("Saved district_master_enriched.csv")
print("Saved district_master_enriched.geojson")

Saved district_master_enriched.csv
Saved district_master_enriched.geojson


# **Qwen Tuning to gain insights and generate flags form dataset**


In [36]:
from kaggle_secrets import UserSecretsClient
secret_label = "HF_TOKEN"
secret_value = UserSecretsClient().get_secret(secret_label)

In [37]:
from kaggle_secrets import UserSecretsClient

try:
    secret = UserSecretsClient().get_secret("HF_TOKEN")
    print("FOUND")
    print(secret[:10])
except Exception as e:
    print("ERROR:")
    print(e)

FOUND
hf_BNwaxIM


In [38]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

hf_token = None

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = os.getenv("HF_TOKEN")

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print("HF login successful.")
else:
    print("HF_TOKEN not found.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login successful.


In [39]:

import torch

print(torch.__version__)
print(torch.cuda.is_available())

2.10.0+cu128
True


In [40]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.3 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22

In [41]:
import bitsandbytes as bnb

print(bnb.__version__)

0.49.2


In [42]:
TEACHER_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    TEACHER_MODEL_NAME,
    token=hf_token,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL_NAME,
    token=hf_token,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto"
)
model.eval()

print("Loaded:", TEACHER_MODEL_NAME)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-7B-Instruct


In [43]:
import json

def qwen_generate(prompt, max_new_tokens=220, temperature=0.2):
    messages = [
        {"role": "system", "content": "You are a disaster operations analyst. Use only the provided structured input. Return concise JSON only."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(out[0], skip_special_tokens=True)

In [44]:
def build_prompt(row):
    return f"""
District record:
district: {row['district']}
state: {row['st_nm']}
flood_risk_score: {row['flood_risk_score']:.2f}
heat_risk_score: {row['heat_risk_score']:.2f}
overall_risk_score: {row['overall_risk_score']:.2f}
water_level_above_danger_m: {row['water_level_above_danger_m']:.2f}
active_flood_hotspots: {int(row['active_flood_hotspots'])}
response_teams_deployed: {int(row['response_teams_deployed'])}
heat_alert_tier: {row['heat_alert_tier']}
flood_severity: {row['flood_severity']}
dual_hazard_flag: {bool(row['dual_hazard_flag'])}

Return JSON with keys:
archetype_name
risk_flags
summary
recommended_actions
"""

In [45]:
insights = []

for _, row in district_final.iterrows():
    prompt = build_prompt(row)
    raw = qwen_generate(prompt)
    insights.append({
        "district_id": row["district_id"],
        "district": row["district"],
        "state": row["st_nm"],
        "raw_qwen_output": raw
    })

insights_df = pd.DataFrame(insights)
display(insights_df.head())

,district_id,district,state,raw_qwen_output
0,D0001,Thane,Maharashtra,system\nYou are a disaster operations analyst....
1,D0002,North Twenty Four Parganas,West Bengal,system\nYou are a disaster operations analyst....
2,D0003,Bangalore,Karnataka,system\nYou are a disaster operations analyst....
3,D0004,Pune,Maharashtra,system\nYou are a disaster operations analyst....
4,D0005,Mumbai Suburban,Maharashtra,system\nYou are a disaster operations analyst....


In [46]:
insights_df.to_json(ART / "district_insights.jsonl", orient="records", lines=True)

# a compact flags file for the dashboard
flags_df = district_final[[
    "district_id", "district", "st_nm",
    "water_level_above_danger_m",
    "active_flood_hotspots",
    "response_teams_deployed",
    "heat_alert_tier",
    "flood_severity",
    "dual_hazard_flag",
    "overall_severity"
]].copy()

flags_df.to_json(ART / "district_flags.jsonl", orient="records", lines=True)

print("Saved district_insights.jsonl")
print("Saved district_flags.jsonl")

Saved district_insights.jsonl
Saved district_flags.jsonl


In [47]:
validation = {
    "district_rows": int(len(district_final)),
    "duplicate_district_state_pairs": int(district_final.duplicated(subset=["district", "st_nm"]).sum()),
    "missing_geometry": int(district_final.geometry.isna().sum()),
    "flood_joined_rows": int(flood_join["district_id"].notna().sum()),
    "flood_unmatched_rows": int(flood_join["district_id"].isna().sum()),
    "weather_matched_rows": int(weather_matched["district"].notna().sum()),
    "overall_severity_counts": district_final["overall_severity"].value_counts(dropna=False).to_dict()
}

with open(ART / "validation_report.json", "w") as f:
    json.dump(validation, f, indent=2)

print(json.dumps(validation, indent=2))

{
  "district_rows": 556,
  "duplicate_district_state_pairs": 0,
  "missing_geometry": 0,
  "flood_joined_rows": 2833,
  "flood_unmatched_rows": 7167,
  "weather_matched_rows": 5112,
  "overall_severity_counts": {
    "Extreme": 497,
    "Low": 50,
    "High": 6,
    "Moderate": 3
  }
}


## What this notebook does

1. Cleans and validates the district GeoJSON backbone.
2. Uses the flood dataset’s lat/lon records as real spatial telemetry and assigns them to districts with `sjoin`.
3. Uses the weather dataset as a heatwave source and maps city-level records to district-level heat risk using exact/alias matches.
4. Computes deterministic flood/heat flags exactly aligned to the assignment’s test cases:
   - cross-disaster overlap
   - water level > 1.5m
   - zero-teams high-severity hotspots
5. Uses Qwen only for grounded explanations and archetype text, not for scoring.
6. Exports the final dashboard artifacts:
   - `district_master_enriched.csv`
   - `district_master_enriched.geojson`
   - `district_flags.jsonl`
   - `district_insights.jsonl`
   - `validation_report.json`